<a href="https://colab.research.google.com/github/23f1000190/Important-colab-or-notebooks/blob/main/Music_Genre_classification_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification

In [2]:
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"priyanshu23f1000190","key":"4cb91d988321b70f3a30e18c67778a30"}'}

In [4]:
!mkdir -p /root/.config/kaggle
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

In [5]:
ls -l /root/.config/kaggle/

total 4
-rw------- 1 root root 75 Feb 28 02:06 kaggle.json


In [6]:
!pip install librosa


In [7]:
import os
import librosa
import torch
import sklearn
import IPython.display as ipd
import matplotlib.pyplot as plt
from google.colab import drive
import os



In [8]:

os.environ["KAGGLE_USERNAME"] = "priyanshu23f1000190"
os.environ["KAGGLE_KEY"] = "4cb91d988321b70f3a30e18c67778a30"

In [9]:
!pip install kagglehub

import kagglehub
path = kagglehub.competition_download("jan-2026-dl-gen-ai-project")
print("Dataset downloaded to:", path)

100%|██████████| 16.1G/16.1G [03:14<00:00, 88.7MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project


In [10]:
print(path)

/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project


In [11]:

base_path = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project"
print(os.listdir(base_path))

['messy_mashup']


In [12]:


base_path = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
print(os.listdir(base_path))

['test.csv', 'mashups', 'genres_stems', 'sample_submission.csv', 'ESC-50-master']


In [13]:

base_path = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
print(os.listdir(base_path))

['blues', 'country', 'rock', 'classical', 'hiphop', 'metal', 'reggae', 'pop', 'jazz', 'disco']


In [14]:
base_path = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/pop"
print(os.listdir(base_path))

['pop.00054', 'pop.00039', 'pop.00085', 'pop.00061', 'pop.00036', 'pop.00077', 'pop.00074', 'pop.00032', 'pop.00072', 'pop.00070', 'pop.00010', 'pop.00009', 'pop.00024', 'pop.00026', 'pop.00016', 'pop.00075', 'pop.00093', 'pop.00095', 'pop.00088', 'pop.00082', 'pop.00048', 'pop.00084', 'pop.00065', 'pop.00064', 'pop.00087', 'pop.00069', 'pop.00011', 'pop.00062', 'pop.00089', 'pop.00021', 'pop.00028', 'pop.00013', 'pop.00000', 'pop.00063', 'pop.00099', 'pop.00001', 'pop.00023', 'pop.00027', 'pop.00002', 'pop.00046', 'pop.00055', 'pop.00019', 'pop.00092', 'pop.00006', 'pop.00020', 'pop.00018', 'pop.00059', 'pop.00008', 'pop.00044', 'pop.00033', 'pop.00042', 'pop.00052', 'pop.00071', 'pop.00014', 'pop.00098', 'pop.00073', 'pop.00005', 'pop.00096', 'pop.00097', 'pop.00035', 'pop.00017', 'pop.00037', 'pop.00031', 'pop.00056', 'pop.00049', 'pop.00076', 'pop.00030', 'pop.00045', 'pop.00012', 'pop.00038', 'pop.00057', 'pop.00066', 'pop.00090', 'pop.00034', 'pop.00050', 'pop.00081', 'pop.00083'

##EDA

In [15]:
base_path = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/pop/pop.00012"
print(os.listdir(base_path))

['bass.wav', 'other.wav', 'drums.wav', 'vocals.wav']


In [17]:
drum_path= "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/pop/pop.00012/drums.wav"

In [18]:
#load the audio
waveform, sr =librosa.load(drum_path, sr=None)
print(waveform)
print(sr)

[ 0.02000427  0.03317261  0.03504944 ... -0.12806702 -0.10917664
 -0.06959534]
44100


In [39]:
#play the audio
ipd.Audio(waveform, rate=sr)

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
'''src_base = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
dst_base = "/content/drive/MyDrive/kaggle_project"

# Create destination folder if it doesn't exist
os.makedirs(dst_base, exist_ok=True)

# Copy ESC-50-master and genres_stems into Drive
!cp -r {src_base}/ESC-50-master {dst_base}/
!cp -r {src_base}/genres_stems {dst_base}/


^C
